In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # this is the only command that stops my kernel from breaking when running the code
import pip
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
import numpy as np
import time
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

path = "C:/Users/empri/.cache/kagglehub/datasets/denizkavi1/brain-tumor/versions/2"

augment_transform = transforms.Compose([ #transform including augmentations to data
    transforms.Resize((128, 128)),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.03, contrast=0.03) # alters brightness and contrast of images - mild due to small dataset
    ], p=0.5),
    transforms.RandomAffine(0, scale=(0.98, 1.02)),#alters how zoomed in the image is
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

no_augment_transform = transforms.Compose([#standard set of transforms to process the data
    transforms.Resize((128, 128)),
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

#loading the dataset
full_dataset = datasets.ImageFolder(root=path)
class_names = full_dataset.classes
print(class_names)

#splitting the dataset into different sections
train_size = int(0.75 * len(full_dataset))
val_size   = int(0.125 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

#training dataset, validation subset and test subset formed
train_subset, val_subset, test_subset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size]
)

#training dataset is used to train the model
train_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root=path, transform=augment_transform), #use no_augment_transform to test without augmentation
    train_subset.indices
)

#validation dataset is used during training to assess loss on unseen images during training
val_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root=path, transform=no_augment_transform),
    val_subset.indices
)

#test dataset is used to check final performance as these images were completely exempt from training process
test_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root=path, transform=no_augment_transform),
    test_subset.indices
)

#data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_dataset, shuffle=False)

class_map = { #assigning each class the name of the tumour
    0: "meningioma",
    1: "glioma",
    2: "pituitary"
}

#printing some of the validation dataset - ensures it was successfully uploaded
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i, (inputs, targets) in enumerate(val_loader):
    ax = axes[i // 5, i % 5]
    ax.imshow(inputs[0].squeeze())
    ax.set_title(class_map[targets[0].item()])
    ax.axis('off')
    if i == 9:
        break

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn

class BrainTumorCNN(nn.Module): #class to build the convolution neural network
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  #3 convolution layers, 1 or 2 underfits the model and 4 takes too long for not much improvement
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) #pooling layer
        self.relu = nn.ReLU()
        
        flattened_size = 128 * 16 * 16  # channels*height*width,
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.4)#increase if overfitted
        
        #fully connected layers
        self.fc1 = nn.Linear(flattened_size, 256) #use 256 for accuracy
        self.fc2 = nn.Linear(256, 64) #the first num must be equal to no. above
        self.fc3 = nn.Linear(64, num_classes)
        
        #loss function
        self.loss_fn = nn.CrossEntropyLoss()
        
        
    def forward(self, x): #forward passes
        x = self.relu(self.conv1(x))#passed through first convolution layer, ReLu activated
        x = self.pool(x)#downssmpled by pool
        
        x = self.relu(self.conv2(x))#second conv layer - deeper features identified
        x = self.pool(x) #followed by pooling layer
        
        x = self.relu(self.conv3(x))#third conv layer - learns more detailed features
        x = self.pool(x)#pooling
        
        x = self.flatten(x)#maps to 1d layer
        
        x = self.relu(self.fc1(x))#first dense layer
        x = self.dropout(x)
        
        x = self.relu(self.fc2(x))#second dense layer
        x = self.dropout(x)
        
        out = self.fc3(x)#final fully connected layer (logits)
        return out
        
    def training_step(self, batch, device=None):#training steps, forward pass then loss calclation
        x, y = batch
        if device is not None:
            x = x.to(device) #move to cpu
            y = y.to(device)
        y_hat = self(x) #forawrd pass
        loss = self.loss_fn(y_hat, y) #calculate crossentropyloss
        return loss #gives the loss

    def predict_single(self, input, target=None, device=None):#final predictions
        self.eval() #evaluating the model
        with torch.no_grad(): #
            x = input.unsqueeze(0) #batch dimension
            if device is not None:
                x = x.to(device)  #move to cpu
            preds = self(x)   #forward pass
            probs = torch.softmax(preds, dim=1)[0].detach().cpu() #probability of each class
            predicted_class = int(torch.argmax(probs))#predicted class - class with largest probability
            
            print("Predicted class index:", predicted_class)#outputting predicted class and true class
            if target is not None:
                print("True class index:", int(target))
            print("Class probabilities:", probs)
        return predicted_class, probs #returns the prediction and probabilities

In [ ]:
import numpy as np
import copy
import torch

class Trainer: #setting up the training of the network

    def __init__(self, model, n_epochs, lr, train_loader, val_loader, device, patience=5, min_delta=0.0):
        self.model = model 
        self.n_epochs = n_epochs #number of epochs- complete training steps
        self.loss_train = np.zeros(n_epochs)  #array to store training loss per epoch
        self.loss_val   = np.zeros(n_epochs)
        self.train_loader = train_loader 
        self.val_loader   = val_loader
        self.device = device
        
        #opitmisation methods - adam optimiser
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr) 
        lambda_func = lambda epoch: 0.925 #multiplies learning rate by 0.925 after every epoch
        self.scheduler = torch.optim.lr_scheduler.MultiplicativeLR(optimizer=self.optimizer,lr_lambda=lambda_func) #learning rate scheduler
        self.model.to(self.device)
 
        self.patience = patience #number of increases in loss before autostop implemented
        self.min_delta = min_delta #minimum 
        self.best_state = None
        self.best_val = float('inf')
        self.bad_epochs = 0

    def fit(self):
        for epoch in range(self.n_epochs):

            #training loss
            self.model.train()
            loss_t = [] #stores loss values for this batch
            for batch in self.train_loader:
                loss = self.model.training_step(batch, device=self.device)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                loss_t.append(loss.item()) #adds loss value to array
            self.loss_train[epoch] = np.mean(loss_t) #the loss for the batch is the average loss


            #validation loss - same process as training set loop
            self.model.eval()
            loss_v = []
            with torch.no_grad():
                for batch in self.val_loader:
                    loss = self.model.training_step(batch, device=self.device).item()
                    loss_v.append(loss)
            self.loss_val[epoch] = np.mean(loss_v)

            #prints the losses of each epoch
            print(f"Epoch [{epoch+1}/{self.n_epochs}]  train_loss: {self.loss_train[epoch]:.4f}  val_loss: {self.loss_val[epoch]:.4f}")

            #early stopping 
            if self.loss_val[epoch] < (self.best_val - self.min_delta): #checks if loss has improved
                self.best_val = self.loss_val[epoch]
                self.best_state = copy.deepcopy(self.model.state_dict())
                self.bad_epochs = 0 #if it is not added as a bad epoch
            else:
                self.bad_epochs += 1 #if loss has worsened, adds to number of bad epochs
                if self.bad_epochs >= self.patience:#if the number of bad epochs exceeds patience, early stop is triggered
                    print(f"Early stopping triggered at epoch {epoch+1}. Best val_loss: {self.best_val:.4f}")
                    break
            self.scheduler.step()

        best_epoch = np.argmin(self.loss_val) #the best epoch based on lowest validation loss
        print( #prints all important values
            f"Best Epoch {best_epoch+1}/{self.n_epochs} | "
            f"Train Loss: {self.loss_train[best_epoch]:.4f} | "
            f"Val Loss: {self.loss_val[best_epoch]:.4f} | "
            f"Final LR: {self.optimizer.param_groups[0]['lr']:.6f}")


        #restore best weights if early stop is triggered
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            print("Restored model weights from best epoch.")

In [ ]:
model = BrainTumorCNN().to(device) #applying model to images

input, target = val_dataset[3] # pre-trained prediction, expected to be completely random
model.predict_single(input, target, device=device)

In [ ]:
trainer = Trainer(model, n_epochs=25, lr=1e-4, train_loader=train_loader, val_loader=val_loader, device=device, patience=5, min_delta=0.0) 
trainer.fit()#training the model

In [ ]:
x = np.arange(0,trainer.n_epochs)

plt.plot(x, trainer.loss_train, label="Training Loss", color = "royalblue")
plt.plot(x, trainer.loss_val, label="Validation Loss", color = "mediumvioletred")
plt.xlim(0,25)
plt.xlabel("Epoch")
plt.title("Training vs Validation Loss")
plt.ylabel("Loss")
plt.legend()
#plt.savefig("ACTUAL FINAL LOSS.png")
plt.show()

In [ ]:
def example_prediction(img, true_label, index=None): #plotting example predictions from test data
    plt.figure(figsize=(3, 3))
    plt.imshow(img)#, cmap='gray')
    plt.title(f"True: {class_map[int(true_label)]}  (index {index})")#gives model prediction vs. actual class
    plt.axis('off')
    plt.show()

input, target = test_dataset[50] #done for every fifty test batch images - test batch not involved in training
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 50)

input, target = test_dataset[100]
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 100)

input, target = test_dataset[150]
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 150)

input, target = test_dataset[200]
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 200)

input, target = test_dataset[250]
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 250)

input, target = test_dataset[300]
model.predict_single(input, target, device=device)
example_prediction(input[0], target, 300) #doing multiple means a high likelihood of seeing both correct and incorrect predictions

In [ ]:
#MODEL PERFORMANCE CALCULATIONS

import matplotlib.colors as colors
import seaborn as sns
from sklearn.metrics import confusion_matrix

#making a confusion matrix
model.eval() 
ys = [] #stores true labels
y_hats = [] #stores predicted labels
with torch.no_grad(): #no gradient calculation
    for batch in test_loader:
        x, y = batch
        x = x.to(device) 
        logits = model(x) #forward pass
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy() #calculate prediction
        ys.extend(y.cpu().numpy().tolist()) #stores true label
        y_hats.extend(pred.tolist()) #stores prediction

confusion = confusion_matrix(ys, y_hats) #calculates confusion matrix
cm_percent = confusion.astype(float) / confusion.sum(axis=1)[:, None] * 100 #normalises the rows and converts to percentages
accuracy = np.trace(confusion) / np.sum(confusion) #accuracy of the model
print(f"Accuracy: {accuracy:.3f}")

plt.figure(figsize=(6, 5)) #plotting confusion matrix as heatmap
sns.heatmap(cm_percent, annot=True, fmt=".2f", cmap='BuPu', xticklabels=["meningioma", "glioma", "pituitary"], yticklabels=["meningioma", "glioma", "pituitary"])
plt.xlabel("Predicted tumour class")
plt.ylabel("True tumour class")
plt.title("Confusion Matrix")
plt.tight_layout()
#plt.savefig("FINAL CONFUSION PLS.png")
plt.show() # the accuracy of the meningioma ranges from 70-85 beacuase the small dataset means the randomly assigned training dataset heavily infuences accuracy
#because of this inconsistency please allow for a slight difference from the matrix in my report



#TRUE POSITIVE AND FALSE POSITIVE RATES
true_positive_rate = np.diag(confusion) / np.sum(confusion, axis=1)#true positive rates - correctly identified tumour
false_positive = np.sum(confusion, axis=0) - np.diag(confusion) #false postives
true_negative = np.sum(confusion) - (np.sum(confusion, axis=1) + np.sum(confusion, axis=0) - np.diag(confusion)) #true negative
false_positive_rate = false_positive / (false_positive + true_negative) # flase positive rates - incorrectly identified tumour

print("True positive rate per class:", true_positive_rate) 
print("False positive rate per class:", false_positive_rate)

#overall TPR (same as accuracy for multi-class)
overall_TPR = np.trace(confusion) / np.sum(confusion)
print(f"Overall true posive rate: {overall_TPR:.3f}")

#overall FPR
overall_FPR = np.sum(false_positive) / (np.sum(false_positive) + np.sum(true_negative))
print(f"Overall false positive rate: {overall_FPR:.3f}")


correct_probs = [] # asked chatgpt if there was anyway to calculate the average probability for correct predicitions and gave the loop below
incorrect_probs = [] # initially the loop only had the correct predictions, i added the lines detailing the prob for incorrect predictions

model.eval() 
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x)
        probs = torch.softmax(preds, dim=1)

        predicted_class = torch.argmax(probs, dim=1)
        correct_mask = predicted_class == y
        incorrect_mask = predicted_class != y 

        #extract probability assigned to the true class
        predicted_probs = probs[torch.arange(len(y)), predicted_class]

        #keep only the ones where prediction was correct
        correct_probs.extend(predicted_probs[correct_mask].cpu().numpy())
        incorrect_probs.extend(predicted_probs[incorrect_mask].cpu().numpy())                    

average_confidence_correct = sum(correct_probs) / len(correct_probs) #average confidence on correct classifications
average_confidence_incorrect = sum(incorrect_probs) / len(incorrect_probs) #average confidence on incorrect classifications
print("Average probability on correct predictions:", average_confidence_correct)
print("Average probability on incorrect predictions:", average_confidence_incorrect)




In [ ]:
#PLOTTING ROC CURVES
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import torch

all_probs = [] #stores all probabilities
all_labels = [] #stores all labels

model.eval() #asked copilot how to indetify all predicted probailites and labels - output the following loop
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)  
        probs = torch.softmax(outputs, dim=1) 

        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

all_probs = torch.cat(all_probs, dim=0).numpy() #stores probabilities
all_labels = torch.cat(all_labels, dim=0).numpy() #stores labels

n_classes = len(set(all_labels)) #number of classes
y_true = label_binarize(all_labels, classes=list(range(n_classes)))

fpr = {} #false positive rate
tpr = {} #true positive rate
roc_auc = {} #area under curve

for i in range(n_classes): #loop to identify ROC  curve and AUC
    fpr[i], tpr[i], _ = roc_curve(y_true[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = ["royalblue","darkmagenta","hotpink"]
plt.figure(figsize=(8, 6))

for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], lw=2,color=colors[i],
             label=f"{class_map[i]} (AUC = {roc_auc[i]:.3f})") #plotting details

plt.plot([0, 1], [0, 1], 'k--', lw=1) # tpr = fpr baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend()
plt.grid(True)
#plt.savefig("ROC Curve")
plt.show()



In [ ]:
#NEW RESNET18 MODEL
from torchvision import models, transforms
train_transform = transforms.Compose([ #more extreme transforms - trained on very varied data
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),  
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) 
])

val_transform = transforms.Compose([ 
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


import torch
import torch.nn as nn
from torchvision import models

class ResNet18(nn.Module): #class for the resnet architecture - mainly editing some layers. 
    def __init__(self, num_classes=3, class_weights=None):
        super().__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        #asked copilot the reccomended edits to the layers of the ResNet18 model for MRI images, output the following code;
        
        old_weights = self.model.conv1.weight.clone() #changing first conv layer
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.conv1.weight.data = old_weights.mean(dim=1, keepdim=True)  #first conv layer accepts greyscale images rather than RGB

        #replacing the classfier
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, num_classes)#means it will output one of the tumour classes

        #freezing backbone except layer 4
        for name, param in self.model.named_parameters():
            if "layer4" in name or "fc" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        #loss calculation
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        #END OF COPILOT RECOMENDATION
    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, device):
        x, y = batch
        x, y = x.to(device), y.to(device)

        logits = self(x)
        loss = self.criterion(logits, y)

        return loss

    
#CAN USE THE SAME TRAINER FOR THE RESNET18 MODEL

In [ ]:
model = ResNet18().to(device) 

In [ ]:
trainer = Trainer(model, n_epochs=25, lr=1e-4, train_loader=train_loader, val_loader=val_loader, device=device, patience=3, min_delta=0.0) 
trainer.fit()#training the model, patience decreased as it is expected to overfit very fast

In [ ]:
epochs_ran = np.where(trainer.loss_val == 0)[0]#loop to fit graph to only the max amount of epochs run
if len(epochs_ran) > 0: #implemeted here bc early stopping always occurred
    last_epoch = epochs_ran[0]
else:
    last_epoch = trainer.n_epochs

x = np.arange(0, last_epoch) #plotting details
plt.figure(figsize=(7,5))
plt.plot(x, trainer.loss_train[:last_epoch], label="Training Loss", color = "royalblue")
plt.plot(x, trainer.loss_val[:last_epoch], label="Validation Loss", color = "mediumvioletred")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#CONFUSION MATRIX
model.eval()
ys = []
y_hats = []
with torch.no_grad():
    for batch in test_loader:
        x, y = batch
        x = x.to(device)
        logits = model(x)
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
        ys.extend(y.cpu().numpy().tolist())
        y_hats.extend(pred.tolist())

cm = confusion_matrix(ys, y_hats)
cm_percent = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
accuracy = np.trace(cm) / np.sum(cm)
print(f"Accuracy: {accuracy:.3f}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm_percent, annot=True, fmt=".2f", cmap='BuPu', xticklabels=["meningioma", "glioma", "pituitary"], yticklabels=["meningioma", "glioma", "pituitary"])
plt.xlabel("Predicted tumour class")
plt.ylabel("True tumour class")
plt.title("Confusion Matrix")
plt.tight_layout()
#plt.savefig("actual resnet confusion matrix.png")
plt.show()


#TRUE POSITIVE AND FALSE POSITIVE RATES

true_positive_rate = np.diag(confusion) / np.sum(confusion, axis=1)#true positive rates - correctly identified tumour
false_positive = np.sum(confusion, axis=0) - np.diag(confusion) #false postives
true_negative = np.sum(confusion) - (np.sum(confusion, axis=1) + np.sum(confusion, axis=0) - np.diag(confusion)) #true negative
false_positive_rate = false_positive / (false_positive + true_negative) # flase positive rates - incorrectly identified tumour

print("True positive rate per class:", true_positive_rate) 
print("False positive rate per class:", false_positive_rate)

#overall TPR
overall_TPR = np.trace(confusion) / np.sum(confusion)
print(f"Overall true posive rate: {overall_TPR:.3f}")

#overall FPR
overall_FPR = np.sum(false_positive) / (np.sum(false_positive) + np.sum(true_negative))
print(f"Overall false positive rate: {overall_FPR:.3f}")



#correct and incorrect confidences

correct_probs = [] # asked chatgpt if there was anyway to calculate the average probability for correct predicitions and gave the loop below
incorrect_probs = [] # initially the loop only had the correct predictions, i added the lines detailing the prob for incorrect predictions

model.eval() 
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x)
        probs = torch.softmax(preds, dim=1)

        predicted_class = torch.argmax(probs, dim=1)
        correct_mask = predicted_class == y
        incorrect_mask = predicted_class != y 

        # extract probability assigned to the true class
        predicted_probs = probs[torch.arange(len(y)), predicted_class]

        # keep only the ones where prediction was correct
        correct_probs.extend(predicted_probs[correct_mask].cpu().numpy())
        incorrect_probs.extend(predicted_probs[incorrect_mask].cpu().numpy())                    

average_confidence_correct = sum(correct_probs) / len(correct_probs)
average_confidence_incorrect = sum(incorrect_probs) / len(incorrect_probs)
print("Average probability on correct predictions:", average_confidence_correct)
print("Average probability on incorrect predictions:", average_confidence_incorrect)